# AI Assisted Referee 
## Introduction

- AI helps us but its answers can't be rely on. **Hallucination is unavoidable.**
- It's a no excuse for industries like law, finance, healthcare, and even aviation.

## Inspiration
- What happened inside a football game.
- I decided to built my own Search Engine in Football Rules based on IFAB's Law of The Game.

## Architecture
![Teks Alternatif](./diagram.png)

## Data Ingestion

In [1]:
import pymupdf

doc = pymupdf.open("Laws of the Game.pdf")

- Page: 115 - 128
- Index: 114 - 127

Font Paragraf: Cambria char flags 16

In [2]:
doc[114].get_text()

'115\nLaws of the Game 2026/27  |  Law 12  |  Fouls and Misconduct\n\x07Direct and indirect free kicks and penalty kicks can only be awarded for\noffences committed when the ball is in play.\n1.\t Direct free kick\n\x07A direct free kick is awarded if a player commits any of the following \noffences against an opponent in a manner considered by the referee to be \ncareless, reckless or using excessive force:\n•\u2002charges\n•\u2002jumps at\n•\u2002kicks or attempts to kick\n•\u2002pushes\n•\u2002strikes or attempts to strike (including head-butt)\n•\u2002tackles or challenges\n•\u2002trips or attempts to trip\n\x07If an offence involves contact, it is penalised by a direct free kick.\x07\n•\u2002\x07Careless is when a player shows a lack of attention or consideration when \nmaking a challenge or acts without precaution. No disciplinary sanction \nis needed\n•\u2002\x07Reckless is when a player acts with disregard to the danger to, \nor consequences for, an opponent and must be caution

In [6]:
doc[115].get_text("dict")

{'width': 419.52801513671875,
 'height': 595.2760009765625,
 'blocks': [{'type': 0,
   'number': 0,
   'flags': 0,
   'bbox': (39.68510055541992,
    564.0064697265625,
    47.11309814453125,
    571.740478515625),
   'lines': [{'spans': [{'size': 6.0,
       'flags': 4,
       'bidi': 0,
       'char_flags': 16,
       'font': 'Roag-Regular',
       'color': 4616322,
       'alpha': 255,
       'ascender': 1.093999981880188,
       'descender': -0.19499999284744263,
       'text': '116',
       'origin': (39.68510055541992, 570.5704956054688),
       'bbox': (39.68510055541992,
        564.0064697265625,
        47.11309814453125,
        571.740478515625)}],
     'wmode': 0,
     'dir': (1.0, 0.0),
     'bbox': (39.68510055541992,
      564.0064697265625,
      47.11309814453125,
      571.740478515625)}]},
  {'type': 0,
   'number': 1,
   'flags': 0,
   'bbox': (53.858299255371094,
    69.1561050415039,
    343.1405029296875,
    171.38510131835938),
   'lines': [{'spans': [{'size':

## Chunk Extraction (Naive Method)

In [7]:
import re

cleaned_chunks = []

def process_page(page_num):
    page_blocks = doc[page_num].get_text("dict")["blocks"]
    cleaned_blocks = []
    
    for b in page_blocks:
        block_text = ""

        if b.get('type') != 0:
            continue

        for line in b["lines"]:
            for span in line["spans"]:
                if span['font'] == 'Cambria' and round(span['size']) == 9:
                    block_text += span['text'] + " "

        block_text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\xff]', '', block_text)
        block_text.replace('\u2002', ' ')
        block_text = re.sub(r'\s+', ' ', block_text).strip()

        if not block_text or block_text.isdigit():
            continue

        cleaned_blocks.append(block_text)

    return cleaned_blocks

# process_page(127)

for i in range(114, 128):
    cleaned_chunks += process_page(i)

cleaned_chunks

['Direct and indirect free kicks and penalty kicks can only be awarded for offences committed when the ball is in play.',
 'A direct free kick is awarded if a player commits any of the following offences against an opponent in a manner considered by the referee to be careless, reckless or using excessive force:',
 '• charges • jumps at • kicks or attempts to kick • pushes • strikes or attempts to strike (including head-butt) • tackles or challenges • trips or attempts to trip',
 'If an offence involves contact, it is penalised by a direct free kick.',
 '• Careless is when a player shows a lack of attention or consideration when making a challenge or acts without precaution. No disciplinary sanction is needed • Reckless is when a player acts with disregard to the danger to, or consequences for, an opponent and must be cautioned • Using excessive force is when a player exceeds the necessary use of force and/or endangers the safety of an opponent and must be sent off',
 'A direct free kic

## Parent-Child Chunk Extraction

In [8]:
final_chunks = []
n = len(cleaned_chunks)
i = 0

while i < n:
    current = cleaned_chunks[i].strip()

    if not current:
        i += 1
        continue

    # 1. CEK APAKAH ELEMEN SAAT INI ADALAH INDUK / CHUNK BARU
    # (Diawali Huruf KAPITAL & BUKAN bullet '•')
    is_parent_start = current[0].isupper() and not current.startswith("•")

    if is_parent_start:

        # Cek apakah elemen setelahnya adalah Bullet Point ('•')
        has_next = i + 1 < n
        next_is_bullet = has_next and cleaned_chunks[i + 1].strip().startswith(
            "•"
        )

        # KASUS A: Kalimat Mandiri Tanpa Anak Bullet
        # (Diakhiri titik '.' DAN elemen berikutnya BUKAN bullet)
        if current.endswith(".") and not next_is_bullet:
            final_chunks.append(current)
            i += 1

        # KASUS B: Induk Kalimat yang Memiliki Anak Bullet / Teks Lanjutan
        else:
            final_sentence = [current]  # Tampung Induk Kalimat dulu
            i += 1

            # Loop menyedot semua anak bullet '•' & teks lanjutannya
            while i < n:
                next_chunk = cleaned_chunks[i].strip()

                if not next_chunk:
                    i += 1
                    continue

                is_bullet = next_chunk.startswith("•")

                # 💡 KUNCI REVISI: Induk Baru terdeteksi jika diawali KAPITAL & BUKAN bullet!
                # (Tidak peduli diakhiri titik '.', titik dua ':', atau 'or by'!)
                is_new_parent = next_chunk[0].isupper() and not is_bullet

                # Kalau ketemu Induk Baru -> STOP SEDOT & BREAK!
                if is_new_parent:
                    break

                # Masukkan anak bullet / teks lanjutan ke dalam wadah
                final_sentence.append(next_chunk)
                i += 1

            # GABUNGKAN INDUK + SEMUA ANAK JADI 1 CHUNK UTUH!
            final_chunks.append("\n".join(final_sentence))

    else:
        # Fallback jika ada teks non-kapital (lampirkan ke chunk sebelumnya)
        if final_chunks:
            final_chunks[-1] += "\n" + current
        else:
            final_chunks.append(current)
        i += 1

print(
    f"🔥 REVISI SAKTI! Dari {len(cleaned_chunks)} elemen -> Jadi {len(final_chunks)} CHUNK UTUH PRESI!"
)

🔥 REVISI SAKTI! Dari 90 elemen -> Jadi 59 CHUNK UTUH PRESI!


In [9]:
for chunk in final_chunks:
    print(chunk)
    print()

Direct and indirect free kicks and penalty kicks can only be awarded for offences committed when the ball is in play.

A direct free kick is awarded if a player commits any of the following offences against an opponent in a manner considered by the referee to be careless, reckless or using excessive force:
• charges • jumps at • kicks or attempts to kick • pushes • strikes or attempts to strike (including head-butt) • tackles or challenges • trips or attempts to trip

If an offence involves contact, it is penalised by a direct free kick.
• Careless is when a player shows a lack of attention or consideration when making a challenge or acts without precaution. No disciplinary sanction is needed • Reckless is when a player acts with disregard to the danger to, or consequences for, an opponent and must be cautioned • Using excessive force is when a player exceeds the necessary use of force and/or endangers the safety of an opponent and must be sent off

A direct free kick is awarded if a p

# Embedding Huggingface

In [10]:
! pip install langchain-huggingface langchain-chroma chromadb sentence-transformers

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device':'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [12]:
documents = [
    Document(page_content=text)
    for text in final_chunks
]

vector_db = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="ifab_law_12_prod_grade",
    persist_directory="./chroma_db"
)

print("Injection Success!")

Injection Success!


#

In [17]:
# What is yellow card called?
# What happen if i throw shirt for celebration and my goal disallowed?
# Why the foul is visible but the game still on?
# If the player make contact with an opponent and he was lack of attention, what his punishment?

query = """
What is yellow card called?
"""

results = vector_db.similarity_search_with_relevance_scores(query, k=2)

print(f"🔍 QUERY: '{query}'\n")
print("=" * 60)
for doc, score in results:
    print(f"🎯 RELEVANCE SCORE: {score:.4f}")
    # print(f"📌 METADATA: Law {doc.metadata['law']} - {doc.metadata['section']} (p. {doc.metadata['page']})")
    print(f"📄 CONTENT:\n{doc.page_content}")
    print("=" * 60)

🔍 QUERY: '
What is yellow card called?
'

🎯 RELEVANCE SCORE: 0.6899
📄 CONTENT:
The yellow card communicates a caution and the red card communicates a sending-off.
🎯 RELEVANCE SCORE: 0.6899
📄 CONTENT:
The yellow card communicates a caution and the red card communicates a sending-off.


In [ ]:
documents = [
    Document(page_content=text)
    for text in chunks
]

vector_db = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="ifab_law_12_raw",
    persist_directory="./chroma_db"
)

In [ ]:
query = """
If the player make contact with an opponent and he was lack of attention, 
what his punishment?
"""

results = vector_db.similarity_search_with_relevance_scores(query, k=2)

print(f"🔍 QUERY: '{query}'\n")
print("=" * 60)
for doc, score in results:
    print(f"🎯 RELEVANCE SCORE: {score:.4f}")
    # print(f"📌 METADATA: Law {doc.metadata['law']} - {doc.metadata['section']} (p. {doc.metadata['page']})")
    print(f"📄 CONTENT:\n{doc.page_content}")
    print("=" * 60)